### Convert your cleaned dataset into structured documents for RAG
Recipe: Tomato Pasta,
Category: meals,
Ingredients: tomato, garlic, pasta,
Instructions:
Boil pasta...

In [2]:
%pip install pandas
import pandas as pd
import ast

# Load cleaned dataset
df = pd.read_csv("recipes_cleaned.csv")

# Convert NER string → list
def parse_ner(ner):
    try:
        return ast.literal_eval(ner)
    except:
        return []

# Create structured document
def create_document(row):
    ingredients = ", ".join(parse_ner(row['NER']))
    
    return f"""
Recipe: {row['title']}
Category: {row['genre']}
Ingredients: {ingredients}

Instructions:
{row['directions']}
"""

# Apply transformation
df['document'] = df.apply(create_document, axis=1)

# Save new dataset
df.to_csv("recipes_with_docs.csv", index=False)

print("Documents created and saved!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Documents created and saved!


### Convert your documents into embeddings and store them in FAISS (Vector DB)

User query → vector → similarity search → relevant recipes

In [3]:
import pandas as pd
%pip install langchain-community faiss-cpu sentence-transformers

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Load dataset
df = pd.read_csv("recipes_with_docs.csv")

# Convert to LangChain documents
documents = [
    Document(page_content=row['document'])
    for _, row in df.iterrows()
]

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vector_db = FAISS.from_documents(documents, embeddings)

# Save locally
vector_db.save_local("recipe_faiss_index")

print("FAISS index created and saved!")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9660\3013664724.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
c:\Users\LENOVO\Documents\AI_RAG\ai_mini_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1864.29it/s]


FAISS index created and saved!
